In [1]:
import os
import sys

sys.path.insert(0, os.path.abspath(".."))  # repository root (not needed after pip install -e .)

import numpy as np

from data_generation import generate_contrasts, generate_design, generate_response
from psi_pt.inference import psi_pt

In [2]:
p, n0, ns, K = 100, 50, 100, 3   # features, target size, size of each source, number of sources
sigma = 1.0
h = 10.0                           # size of the source contrasts
n_jobs = 4                         # processes for the truncation region

In [3]:
rng = np.random.default_rng(0)

beta0 = np.zeros(p)
beta0[:5] = 2.0                    # target coefficients: 5 true features

X0, X_list = generate_design(rng, p, n0, ns, K)
contrasts = generate_contrasts(rng, p, K, h)
y0, y_list = generate_response(rng, X0, X_list, sigma, beta0, contrasts)

# noise covariances of the target and of each source
Sigma_0 = sigma ** 2 * np.eye(n0)
Sigma_K = [sigma ** 2 * np.eye(ns) for _ in range(K)]

In [4]:
result = psi_pt(X0, y0, X_list, y_list, Sigma_0, Sigma_K, V=3, n_jobs=n_jobs)

print("selected features:", result["selected_model"])
print("branch:", result["branch"])

selected features: [ 0  1  2  3  4 20 34]
branch: transmission


In [5]:
for r in result["results"]:
    truth = "true" if beta0[r["feature"]] != 0 else "null"
    print(f"feature {r['feature']:3d} ({truth})   p-value = {r['p_value']:.4f}")

feature   0 (true)   p-value = 0.0000
feature   1 (true)   p-value = 0.0000
feature   2 (true)   p-value = 0.0000
feature   3 (true)   p-value = 0.0000
feature   4 (true)   p-value = 0.0000
feature  20 (null)   p-value = 0.2550
feature  34 (null)   p-value = 0.8094


In [6]:
# truncation region of the first test: union of intervals of the test statistic
r = result["results"][0]
print(f"feature {r['feature']}: test statistic = {r['test_statistic']:.4f}")
for left, right in r["intervals"]:
    print(f"  [{left:.4f}, {right:.4f}]")

feature 0: test statistic = 1.9392
  [1.0467, 3.3942]
